# PrimeKV — Quick Test Notebook

Run these cells top-to-bottom to install, test, and interactively
compare KV cache strategies. Works on CPU (free tier) or GPU.

**From your phone:** just tap each cell and hit the play button.

## 1. Clone and install

In [ ]:
!git clone https://github.com/arunvenkatadri/PrimeKV.git
%cd PrimeKV
!git checkout claude/scaffold-primekv-Q9QKN
!pip install -e ".[dev,web]" -q

## 2. Run unit tests (no network, no GPU, ~3 seconds)

In [ ]:
!pytest tests/ -v

## 3. Run the comparison CLI with real GPT-2

Downloads GPT-2 (124M) on first run (~500 MB). Takes 30-60s on CPU.

In [ ]:
!python benchmarks/compare.py --model gpt2 --decode-tokens 16 --max-length 64

## 4. Run comparison from Python (more control)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from primekv.eval import Workload, run_comparison
from primekv.cache import PrimeKVCache
from primekv.classifier import RuleBasedClassifier, Tier
from primekv.baselines import FullCache, H2OCache, UniformQuantCache

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

num_layers = model.config.n_layer

caches = {
    "full":        FullCache(num_layers),
    "uniform_int4": UniformQuantCache(num_layers, bits=4),
    "h2o":         H2OCache(num_layers, capacity=32),
    "primekv":     PrimeKVCache(
        num_layers=num_layers,
        classifier=RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3),
        max_entries_per_tier={Tier.SUPPORTING: 32},
    ),
}

workload = Workload(
    prompt="System: you are a helpful assistant. User: What is the capital of France? Assistant:",
    decode_tokens=16,
    max_length=64,
)

report = run_comparison(caches, workload, model, tok)
print(report.to_markdown())
print()
for r in report.results:
    if r.generated:
        print(f"--- {r.name} ---")
        print(r.generated)
        print()

## 5. Launch interactive Gradio UI

This creates a **public share link** you can open in any browser tab
(or send to a collaborator). The link is active as long as this cell
is running.

In [ ]:
from webui.app import build_demo

demo = build_demo()
demo.launch(share=True)

## 6. Inspect PrimeKV tier distribution

In [ ]:
from primekv.metrics import tier_distribution, summarize_stats

# Use the primekv cache from step 4 (still in memory)
pkv = caches["primekv"]

print("Tier distribution:")
for tier, count in tier_distribution(pkv).items():
    print(f"  {tier:12s}  {count} tokens")

print("\nCache stats:")
for k, v in summarize_stats(pkv.stats).items():
    print(f"  {str(k):20s}  {v}")